<a href="https://colab.research.google.com/github/prasannashrestha011/KnowledgeGraph/blob/main/KnowledgeGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
%pip install -U spacy
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_trf
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 43.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 2.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 790.2 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('

In [23]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load('en_core_web_lg')




In [35]:
doc = nlp("""
Retrieval-Augmented Generation (RAG) has increasingly emerged as a robust and practically viable paradigm for mitigating the pervasive issue of hallucinations in large-scale language models by systematically grounding their generative outputs in externally curated knowledge repositories.
Rather than depending exclusively on the inherently limited and static parametric memory encoded within the neural weights of a model, a RAG-based architecture leverages a dynamic retrieval mechanism, wherein semantically pertinent documents are extracted from high-dimensional vector databases, specialized knowledge graphs, or meticulously indexed corpora, subsequently conditioning the generative model on this corroborative evidence to ensure both contextual fidelity and factual precision.
This hybridized methodology has demonstrably enhanced multiple facets of model performance, including, but not limited to, factual accuracy, domain generalization, computational scalability, and adaptability across heterogeneous knowledge domains within real-world production environments.
Recent empirical studies further underscore that the efficacy of RAG pipelines is highly contingent upon meticulous optimization of retrieval processes, encompassing sophisticated chunking strategies to manage information granularity, advanced indexing paradigms that balance retrieval efficiency with semantic coverage, and the deployment of reranking models that selectively attenuate the influence of noisy or contextually irrelevant passages prior to final generative synthesis.
Consequently, RAG has ascended to become a foundational architecture for enterprise-grade question answering systems, knowledge-intensive applications, and continuous learning frameworks, particularly in scenarios necessitating seamless integration of freshly acquired information without extensive retraining.
Moreover, ongoing research posits that synergistic combinations of RAG with techniques such as instruction-tuned transformers, reinforcement learning from human feedback (RLHF), and multi-hop reasoning frameworks may further amplify both the precision and interpretability of model outputs, thereby setting a precedent for next-generation intelligent systems capable of robust, contextually grounded, and human-aligned language understanding.
""")


def extract_entites(sent):
  entities_pairs=[]
  for sent in doc.sents:
    subjects=[]
    objects=[]
    for tok in sent.noun_chunks:
        if tok.root.dep_ in ("nsubj","nsubjpass"):
           subjects.append(tok.text)
        if tok.root.dep_ in ("dobj","pobj","attr"):
           objects.append(tok.text)
    if subjects and objects:
       entities_pairs.append((subjects[0],objects[0]))
  return entities_pairs



In [36]:
def get_relation(sent):
  doc = nlp(sent.text)
  # Matcher class object
  matcher = Matcher(nlp.vocab)

  #define the pattern
  pattern = [{'DEP':'ROOT'},
            {'DEP':'prep','OP':"?"},
            {'DEP':'agent','OP':"?"},
            {'POS':'ADJ','OP':"?"}]

  matcher.add("matching_1",[pattern])

  matches = matcher(doc)
  k = len(matches) - 1

  span = doc[matches[k][1]:matches[k][2]]

  return(span.text)


In [37]:
entities_pair=extract_entites(doc)
relations=[get_relation(sent) for sent in doc.sents]
for sent in doc.sents:
  print(sent)
print("\n")
print(entities_pair)
print("\n")
print(relations)


Retrieval-Augmented Generation (RAG) has increasingly emerged as a robust and practically viable paradigm for mitigating the pervasive issue of hallucinations in large-scale language models by systematically grounding their generative outputs in externally curated knowledge repositories. 

Rather than depending exclusively on the inherently limited and static parametric memory encoded within the neural weights of a model, a RAG-based architecture leverages a dynamic retrieval mechanism, wherein semantically pertinent documents are extracted from high-dimensional vector databases, specialized knowledge graphs, or meticulously indexed corpora, subsequently conditioning the generative model on this corroborative evidence to ensure both contextual fidelity and factual precision. 

This hybridized methodology has demonstrably enhanced multiple facets of model performance, including, but not limited to, factual accuracy, domain generalization, computational scalability, and adaptability acr

In [38]:
def get_subject_verb_object(sent):
    roots = [token for token in sent if token.dep_ == "ROOT" and token.pos_ in ("VERB", "AUX")]
    triples = []

    for root in roots:
        # Subjects
        subjects = [chunk for chunk in sent.noun_chunks
                    if chunk.root.dep_ in ("nsubj", "nsubjpass") and chunk.root.head == root]
        if not subjects:
            continue
        subject = subjects[0].text

        # Relation (verb + auxiliaries)
        auxiliaries = [child for child in root.children if child.dep_ in ("aux", "auxpass", "neg")]
        aux_text = " ".join([aux.text for aux in sorted(auxiliaries, key=lambda x: x.i)])
        relation = " ".join([aux_text, root.text]).strip() if aux_text else root.text

        # Collect object phrases as TEXT (not generators)
        obj_parts = []

        def add_subtree_text(token):
            return " ".join([t.text for t in token.subtree])

        for child in root.children:
            if child.dep_ in ("dobj", "attr", "ccomp", "xcomp"):
                obj_parts.append(add_subtree_text(child))
            elif child.dep_ == "prep":
                obj_parts.append(add_subtree_text(child))

        # Handle conjunctions (e.g., "retrieves X and conditions Y")
        for conj in root.conjuncts:
            for child in conj.children:
                if child.dep_ in ("dobj", "attr", "ccomp", "xcomp"):
                    obj_parts.append(add_subtree_text(child))
                elif child.dep_ == "prep":
                    obj_parts.append(add_subtree_text(child))

        if obj_parts:
            obj_text = " ".join(obj_parts)
            triples.append((subject, relation, obj_text))
        else:
            # Fallback: sometimes the complement is an adjectival or prepositional phrase not captured
            # We can try to get everything after the verb
            verb_idx = root.i - sent.start
            if verb_idx + 1 < len(sent):
                rest = sent[verb_idx + 1:].text
                if rest.strip():
                    triples.append((subject, relation, rest.strip()))

    return triples
all_triples = []
for sent in doc.sents:
    triples = get_subject_verb_object(sent)
    all_triples.extend(triples)

# Print results
for subj, rel, obj in all_triples:
    if obj:  # skip empty objects
        print(f"({subj}, {rel}, {obj})")

(
Retrieval-Augmented Generation (RAG, has emerged, as a robust and practically viable paradigm for mitigating the pervasive issue of hallucinations in large - scale language models by systematically grounding their generative outputs in externally curated knowledge repositories)
(a RAG-based architecture, leverages, Rather than depending exclusively on the inherently limited and static parametric memory encoded within the neural weights of a model a dynamic retrieval mechanism , wherein semantically pertinent documents are extracted from high - dimensional vector databases , specialized knowledge graphs , or meticulously indexed corpora)
(This hybridized methodology, has enhanced, multiple facets of model performance , including , but not limited to , factual accuracy , domain generalization , computational scalability , and adaptability across heterogeneous knowledge domains within real - world production environments)
(Recent empirical studies, underscore, that the efficacy of RAG p

In [42]:
import spacy
from spacy.matcher import Matcher
import re
from collections import defaultdict
from typing import List, Tuple, Set

nlp = spacy.load("en_core_web_sm")

class RelationshipExtractor:
    """Generalized relationship extraction system for any domain"""

    def __init__(self):
        self.nlp = nlp
        self.matcher = Matcher(nlp.vocab)
        self._setup_patterns()

    def _setup_patterns(self):
        """Setup general linguistic patterns for relationship extraction"""
        # Pattern 1: [SUBJECT] [VERB] [OBJECT]
        pattern1 = [
            {"POS": "NOUN", "OP": "+"},
            {"POS": "VERB", "OP": "+"},
            {"POS": "NOUN", "OP": "+"}
        ]

        # Pattern 2: [SUBJECT] [VERB] [PREP] [OBJECT]
        pattern2 = [
            {"POS": "NOUN", "OP": "+"},
            {"POS": "VERB", "OP": "+"},
            {"POS": "ADP"},
            {"POS": "NOUN", "OP": "+"}
        ]

        # Pattern 3: [SUBJECT] [AUX] [VERB] [OBJECT]
        pattern3 = [
            {"POS": "NOUN", "OP": "+"},
            {"POS": "AUX", "OP": "*"},
            {"POS": "VERB", "OP": "+"},
            {"POS": "NOUN", "OP": "+"}
        ]

        self.matcher.add("SUBJ_VERB_OBJ", [pattern1, pattern2, pattern3])

    def extract_relationships(self, text: str) -> List[Tuple[str, str, str]]:
        """Main method to extract relationships from text"""
        doc = self.nlp(text)

        # Use multiple extraction strategies
        dependency_triples = self._extract_dependency_relationships(doc)
        pattern_triples = self._extract_pattern_relationships(doc)
        semantic_triples = self._extract_semantic_relationships(doc)

        # Combine all methods
        all_triples = dependency_triples + pattern_triples + semantic_triples

        # Advanced filtering and normalization
        filtered_triples = self._filter_relationships(all_triples)
        normalized_triples = self._normalize_relationships(filtered_triples)
        final_triples = self._deduplicate_relationships(normalized_triples)

        return final_triples

    def _extract_dependency_relationships(self, doc) -> List[Tuple[str, str, str]]:
        """Extract relationships using dependency parsing"""
        triples = []

        for sent in doc.sents:
            sent_triples = self._process_sentence(sent)
            triples.extend(sent_triples)

        return triples

    def _process_sentence(self, sent) -> List[Tuple[str, str, str]]:
        """Process a single sentence for relationship extraction"""
        triples = []

        # Build entity map for the sentence
        entity_map = self._build_entity_map(sent)

        # Find verbs and their arguments
        for token in sent:
            if self._is_meaningful_verb(token):
                subjects = self._extract_subjects(token, entity_map)
                objects = self._extract_objects(token, entity_map)

                # Generate triples from combinations
                for subj in subjects:
                    for obj in objects:
                        if self._is_valid_triple(subj, token.lemma_, obj):
                            triples.append((subj, token.lemma_, obj))

        return triples

    def _build_entity_map(self, sent) -> dict:
        """Build a map of token positions to entity texts"""
        entity_map = {}

        # Use noun chunks as base entities
        for chunk in sent.noun_chunks:
            clean_entity = self._normalize_entity(chunk.text)
            if self._is_valid_entity(clean_entity):
                entity_map[chunk.root.i] = clean_entity

        # Add named entities
        for ent in sent.ents:
            clean_entity = self._normalize_entity(ent.text)
            if self._is_valid_entity(clean_entity):
                entity_map[ent.start] = clean_entity

        # Fill gaps with individual tokens
        for token in sent:
            if token.i not in entity_map and token.pos_ in ["NOUN", "PROPN"]:
                clean_entity = self._normalize_entity(token.text)
                if self._is_valid_entity(clean_entity):
                    entity_map[token.i] = clean_entity

        return entity_map

    def _extract_subjects(self, verb_token, entity_map) -> List[str]:
        """Extract subjects for a verb"""
        subjects = []

        # Direct subjects
        for child in verb_token.children:
            if child.dep_ in ["nsubj", "nsubjpass"]:
                subject = self._get_entity_text(child, entity_map)
                if subject:
                    subjects.append(subject)

        # Conjoined subjects
        for child in verb_token.children:
            if child.dep_ == "conj":
                for conj_child in child.children:
                    if conj_child.dep_ in ["nsubj", "nsubjpass"]:
                        subject = self._get_entity_text(conj_child, entity_map)
                        if subject:
                            subjects.append(subject)

        return list(set(subjects))

    def _extract_objects(self, verb_token, entity_map) -> List[str]:
        """Extract objects for a verb"""
        objects = []

        # Direct objects
        for child in verb_token.children:
            if child.dep_ in ["dobj", "attr"]:
                obj = self._get_entity_text(child, entity_map)
                if obj:
                    objects.append(obj)

        # Prepositional objects
        for child in verb_token.children:
            if child.dep_ == "prep":
                for prep_child in child.children:
                    if prep_child.dep_ == "pobj":
                        obj = self._get_entity_text(prep_child, entity_map)
                        if obj:
                            objects.append(obj)

        # Clausal complements
        for child in verb_token.children:
            if child.dep_ in ["ccomp", "xcomp"]:
                obj = self._extract_clausal_object(child, entity_map)
                if obj:
                    objects.append(obj)

        return list(set(objects))

    def _extract_clausal_object(self, clause_root, entity_map) -> str:
        """Extract object from clausal complement"""
        # Look for the main entity in the clause
        for token in clause_root.children:
            if token.dep_ in ["dobj", "attr", "nsubj"]:
                return self._get_entity_text(token, entity_map)

        # If no direct object, return the clause itself
        return self._normalize_entity(clause_root.text)

    def _get_entity_text(self, token, entity_map) -> str:
        """Get entity text from entity map or extract from token"""
        if token.i in entity_map:
            return entity_map[token.i]

        # Extract entity dynamically
        entity_text = self._extract_complete_entity(token)
        clean_entity = self._normalize_entity(entity_text)

        if self._is_valid_entity(clean_entity):
            return clean_entity

        return None

    def _extract_complete_entity(self, token) -> str:
        """Extract complete entity starting from a token"""
        start = token.i
        end = token.i + 1

        # Expand left through modifiers
        while start > token.sent.start:
            prev_token = token.doc[start - 1]
            if prev_token.dep_ in ["compound", "amod", "det", "nummod", "nmod"]:
                start -= 1
            else:
                break

        # Expand right through compounds
        while end < token.sent.end:
            next_token = token.doc[end]
            if next_token.dep_ in ["compound", "amod"]:
                end += 1
            else:
                break

        return token.doc[start:end].text

    def _extract_pattern_relationships(self, doc) -> List[Tuple[str, str, str]]:
        """Extract relationships using pattern matching"""
        triples = []
        matches = self.matcher(doc)

        for match_id, start, end in matches:
            span = doc[start:end]

            # Extract components from the pattern
            components = self._analyze_pattern_span(span)
            if components and self._is_valid_triple(*components):
                triples.append(components)

        return triples

    def _analyze_pattern_span(self, span):
        """Analyze pattern span to extract subject, relation, object"""
        # Simple heuristic: first noun chunk is subject, verb is relation, last noun chunk is object
        nouns = [token for token in span if token.pos_ in ["NOUN", "PROPN"]]
        verbs = [token for token in span if token.pos_ == "VERB"]

        if len(nouns) >= 2 and verbs:
            subject = self._normalize_entity(nouns[0].text)
            relation = verbs[0].lemma_
            obj = self._normalize_entity(nouns[-1].text)

            return (subject, relation, obj)

        return None

    def _extract_semantic_relationships(self, doc) -> List[Tuple[str, str, str]]:
        """Extract relationships using semantic patterns"""
        triples = []

        semantic_patterns = [
            # X verb Y pattern
            (r'(\b[A-Z][a-z]+(?:\s+[A-Za-z]+)*)\s+(\w+ed|\w+ing|\w+s)\s+(\b[A-Z][a-z]+(?:\s+[A-Za-z]+)*)'),
            # X verb from/to/for Y
            (r'(\b[A-Z][a-z]+(?:\s+[A-Za-z]+)*)\s+(\w+ed|\w+ing|\w+s)\s+(?:from|to|for|in|on)\s+(\b[A-Z][a-z]+(?:\s+[A-Za-z]+)*)'),
        ]

        for sent in doc.sents:
            sent_text = sent.text
            for pattern in semantic_patterns:
                matches = re.finditer(pattern, sent_text)
                for match in matches:
                    subject = self._normalize_entity(match.group(1))
                    relation = self._normalize_verb(match.group(2))
                    obj = self._normalize_entity(match.group(3))

                    if self._is_valid_triple(subject, relation, obj):
                        triples.append((subject, relation, obj))

        return triples

    def _normalize_entity(self, text: str) -> str:
        """Normalize entity text"""
        if not text:
            return text

        # Clean text
        text = re.sub(r'^\W+|\W+$', '', text)
        text = re.sub(r'\s+', ' ', text).strip()

        # Remove articles and some determiners
        text = re.sub(r'^(a|an|the|some)\s+', '', text, flags=re.IGNORECASE)

        return text

    def _normalize_verb(self, text: str) -> str:
        """Normalize verb to base form"""
        doc = self.nlp(text)
        if doc:
            return doc[0].lemma_
        return text

    def _is_meaningful_verb(self, token) -> bool:
        """Check if verb is meaningful for relationships"""
        weak_verbs = {"be", "have", "do", "get", "make", "become", "seem", "appear"}
        return (token.pos_ == "VERB" and
                token.lemma_ not in weak_verbs and
                token.dep_ in ["ROOT", "conj", "acl", "relcl"])

    def _is_valid_entity(self, text: str) -> bool:
        """Check if entity is valid"""
        if not text or len(text) < 2:
            return False

        vague_terms = {
            'it', 'this', 'that', 'which', 'what', 'there', 'one',
            'result', 'way', 'method', 'approach', 'thing', 'something',
            'everything', 'nothing', 'someone', 'everyone'
        }

        if text.lower() in vague_terms:
            return False

        # Check if it's mostly punctuation or numbers
        if re.match(r'^[\W\d]+$', text):
            return False

        return True

    def _is_valid_triple(self, subject: str, relation: str, obj: str) -> bool:
        """Validate if triple is meaningful"""
        if not all([subject, relation, obj]):
            return False

        if subject.lower() == obj.lower():
            return False

        if len(subject.split()) > 5 or len(obj.split()) > 5:
            return False

        return True

    def _filter_relationships(self, triples: List[Tuple]) -> List[Tuple]:
        """Filter out low-quality relationships"""
        filtered = []

        for subj, rel, obj in triples:
            # Filter based on entity quality
            if (self._is_valid_entity(subj) and
                self._is_valid_entity(obj) and
                len(rel) > 2):  # Relation should be substantial
                filtered.append((subj, rel, obj))

        return filtered

    def _normalize_relationships(self, triples: List[Tuple]) -> List[Tuple]:
        """Normalize relationship representations"""
        normalized = []

        for subj, rel, obj in triples:
            norm_subj = self._normalize_entity(subj)
            norm_obj = self._normalize_entity(obj)
            norm_rel = self._normalize_verb(rel)

            normalized.append((norm_subj, norm_rel, norm_obj))

        return normalized

    def _deduplicate_relationships(self, triples: List[Tuple]) -> List[Tuple]:
        """Remove duplicate relationships"""
        seen = set()
        unique = []

        for triple in triples:
            # Create normalized key
            key = (
                triple[0].lower().strip(),
                triple[1].lower().strip(),
                triple[2].lower().strip()
            )

            if key not in seen:
                seen.add(key)
                unique.append(triple)

        return unique

    def print_results(self, triples: List[Tuple], title: str = "RELATIONSHIP EXTRACTION RESULTS"):
        """Print formatted results"""
        print(f"\n{title}")
        print("=" * 60)

        for i, (subj, rel, obj) in enumerate(triples, 1):
            print(f"{i:2d}. {subj:30} -- {rel:12} --> {obj}")

        print(f"\nTotal relationships: {len(triples)}")

        # Statistics
        subjects = set(s for s, r, o in triples)
        relations = set(r for s, r, o in triples)
        objects = set(o for s, r, o in triples)

        print("\n" + "=" * 60)
        print("SUMMARY:")
        print(f"- Unique subjects: {len(subjects)}")
        print(f"- Unique relations: {len(relations)}")
        print(f"- Unique objects: {len(objects)}")


# Example usage and testing
if __name__ == "__main__":
    # Initialize extractor
    extractor = RelationshipExtractor()

    # Sample texts from different domains
    sample_texts = [
        # Technology domain
        """
     Retrieval-Augmented Generation (RAG) has increasingly emerged as a robust and practically viable paradigm for mitigating the pervasive issue of hallucinations in large-scale language models by systematically grounding their generative outputs in externally curated knowledge repositories.
Rather than depending exclusively on the inherently limited and static parametric memory encoded within the neural weights of a model, a RAG-based architecture leverages a dynamic retrieval mechanism, wherein semantically pertinent documents are extracted from high-dimensional vector databases, specialized knowledge graphs, or meticulously indexed corpora, subsequently conditioning the generative model on this corroborative evidence to ensure both contextual fidelity and factual precision.
This hybridized methodology has demonstrably enhanced multiple facets of model performance, including, but not limited to, factual accuracy, domain generalization, computational scalability, and adaptability across heterogeneous knowledge domains within real-world production environments.
Recent empirical studies further underscore that the efficacy of RAG pipelines is highly contingent upon meticulous optimization of retrieval processes, encompassing sophisticated chunking strategies to manage information granularity, advanced indexing paradigms that balance retrieval efficiency with semantic coverage, and the deployment of reranking models that selectively attenuate the influence of noisy or contextually irrelevant passages prior to final generative synthesis.
Consequently, RAG has ascended to become a foundational architecture for enterprise-grade question answering systems, knowledge-intensive applications, and continuous learning frameworks, particularly in scenarios necessitating seamless integration of freshly acquired information without extensive retraining.
Moreover, ongoing research posits that synergistic combinations of RAG with techniques such as instruction-tuned transformers, reinforcement learning from human feedback (RLHF), and multi-hop reasoning frameworks may further amplify both the precision and interpretability of model outputs, thereby setting a precedent for next-generation intelligent systems capable of robust, contextually grounded, and human-aligned language understanding.
        """
    ]

    # Test on each domain
    domains = ["Technology", "Healthcare", "Education"]

    for i, (text, domain) in enumerate(zip(sample_texts, domains)):
        print(f"\n{'='*80}")
        print(f"DOMAIN: {domain}")
        print(f"{'='*80}")

        relationships = extractor.extract_relationships(text)
        extractor.print_results(relationships, f"{domain.upper()} RELATIONSHIPS")

    # Your specific RAG text
    rag_text = """Retrieval-Augmented Generation (RAG) has emerged as a practical method for reducing hallucinations in large language models by grounding their outputs in external knowledge sources. Instead of relying solely on parametric memory, a RAG system retrieves relevant documents from a vector database or search index and conditions the model on this evidence during generation. This hybrid approach has been shown to significantly improve factual accuracy, scalability, and domain adaptability in production systems. Recent research highlights that RAG pipelines benefit from careful retrieval optimization, including chunking strategies, indexing methods, and reranking models that filter noisy passages before final generation. As a result, RAG has become one of the most widely adopted architectures for enterprise-level question answering, data-intensive applications, and continuous knowledge integration."""

    print(f"\n{'='*80}")
    print("SPECIFIC TEST: RAG TEXT")
    print(f"{'='*80}")

    rag_relationships = extractor.extract_relationships(rag_text)
    extractor.print_results(rag_relationships, "RAG-RELATED RELATIONSHIPS")


DOMAIN: Technology

TECHNOLOGY RELATIONSHIPS
 1. Retrieval-Augmented Generation -- emerge       --> robust and practically viable paradigm
 2. RAG-based architecture         -- leverage     --> dynamic retrieval mechanism
 3. wherein semantically pertinent documents -- extract      --> high-dimensional vector databases
 4. This hybridized methodology    -- enhance      --> multiple facets
 5. Recent empirical studies       -- underscore   --> efficacy
 6. advanced indexing              -- underscore   --> efficacy
 7. advanced indexing              -- paradigm     --> that balance retrieval efficiency
 8. RAG                            -- ascend       --> foundational architecture
 9. ongoing research posits        -- amplify      --> both the precision

Total relationships: 9

SUMMARY:
- Unique subjects: 8
- Unique relations: 8
- Unique objects: 8

SPECIFIC TEST: RAG TEXT

RAG-RELATED RELATIONSHIPS
 1. Retrieval-Augmented Generation -- emerge       --> practical method
 2. RAG system